# Tier 13b: machines from a supplier database, and gearbox stage count

Plan 033 refines Tier 13 (plan 018). The Halo reference picks a rotor reduction of about 36:1, with motors at about
12,900 rpm. Two reasons:

- the Tier 13 machine mass uses one torque density (15 N m/kg) at every speed;
- the AFDD drive weight barely depends on the reduction ratio.

This notebook verifies two interchangeable options and shows what the optimizer does with them.

| Piece | Model | Source |
|---|---|---|
| Machine data | `data/machines/aerospace_motors.csv`: 21 machines, peak and continuous ratings, inverter status, one URL per row | Evolito, Helix, H3X, magniX, Siemens, Safran, EMRAX (URLs in the CSV) |
| `DatabaseMassModel` | continuous torque density tau = tau_ref (w / 500 rad/s)^-a at base speed w = P / T_cont, least-squares fit; cap of 20 kW/kg; optional inverter term; relaxed stack count | the fitted rows of the database |
| `GearStageModel` | stages = smooth ceil(ln r / ln 5); efficiency 0.99 x 0.99^n; mass factor 1 + 0.3 (n - 1), normalised at the XV-15 ratio of 35.4:1 | NASA TP-2795 (planetary stage 99.44-99.75 %), NASA TM-81426 (spur > 98 %) |

The switches are `HaloAssumptions.machine_mass_model` (`"torque_density"` by default, or `"database"`) and
`HaloAssumptions.gearbox_stages` (default False). The defaults reproduce the 14,037 lb reference; the unit test
suite, run at the end of this notebook, asserts it.

Trades use the fast set: `assumptions_plan027` (thermal off) with Scholz aerodynamics, about 10-60 s per solve.
Section 6 sizes the full reference with both options on (thermal, AeroBuildup), which takes several minutes.

In [ ]:
import sys
import time
from dataclasses import replace
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
for path in (repo_root / "src", repo_root):     # this checkout's package even if another one is installed
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import aerosandbox as asb
import aerosandbox.numpy as np
import aerosandbox.tools.units as u
import matplotlib.pyplot as plt

from aircraft_closure.powertrain.components.gearbox import GearStageModel
from aircraft_closure.powertrain.components.motor import DatabaseMassModel, Motor, TorqueDensityMassModel, rubber_machine
from aircraft_closure.powertrain.machine_database import fit_torque_density, load_machine_database
from aircraft_closure.weights import afdd
from examples.halo_sizing import (HaloAssumptions, assumptions_plan027, ratio_reference_gear_stages, solve_halo_sizing,
                                  stage_mass_ratio)

# Plan 035 changed the defaults (drag corrections, redundancy, unit-built machines, gearbox stages); this notebook
# keeps reproducing its own tier on the plan 030 settings.
from functools import partial
from examples.halo_sizing import HaloAssumptions as _HaloAssumptions, solve_halo_sizing as _solve_halo_sizing
from examples.halo_sizing import pre_plan035 as _pre_plan035
HaloAssumptions = partial(_HaloAssumptions, **_pre_plan035)
solve_halo_sizing = partial(_solve_halo_sizing, assumptions=HaloAssumptions())
from examples.xv15_reference import calibration_factors

check_log = []


def check(name, passed):
    # Record and print one named verification.
    passed = bool(passed)
    check_log.append((name, passed))
    print(("PASS  " if passed else "FAIL  ") + name)


ink, ink_muted, grid_color = "#1a1a19", "#5e5d58", "#e4e3dc"
blue, orange, aqua, yellow, violet = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cd6"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": grid_color, "grid.linewidth": 0.8,
                     "axes.labelcolor": ink, "xtick.color": ink_muted, "ytick.color": ink_muted,
                     "axes.titlesize": 11, "axes.titleweight": "bold", "font.size": 9.5})
rpm = u.rpm
ratios = dict(torque_ratio=2.5, power_ratio=1.25, speed_ratio=2.5)    # the Halo rubber machine
fast = replace(assumptions_plan027, aerodynamics_model="scholz")

## 1. The database and the torque-density fit

The speed class of a machine is its base speed, w = P_cont / T_cont. For the Halo `rubber_machine` this is the
peak-efficiency speed: continuous torque = 0.5 x peak torque.

The fit uses bare machines, or machines whose inverter status is unknown, that publish continuous torque and
power. Two kinds of rows are plotted and compared but not fitted:

- integrated rows (inverter included): the comparison adds an inverter at 20 kW/kg;
- peak-only rows: the comparison takes continuous = 0.5 x peak.

In [ ]:
records = load_machine_database()
fit = fit_torque_density(records)
residual = dict(fit.residuals)
print(f"{'machine':22s} {'type':7s} {'inv':7s} {'mass kg':>8s} {'T_cont':>7s} {'T_peak':>7s} {'P_cont kW':>9s} "
      f"{'base rpm':>8s} {'N m/kg':>7s} {'kW/kg':>6s} {'fit':>4s} {'model/db':>8s}")
for r in records:
    w = r.speed_base_rad_s()
    tau = r.torque_density_continuous_Nm_kg()
    fmt = lambda v, f: (f % v) if v is not None else "-"
    print(f"{r.name:22s} {r.topology:7s} {r.inverter_included:7s} {r.mass_kg:8.1f} {fmt(r.torque_continuous_Nm, '%7.0f'):>7s} "
          f"{fmt(r.torque_peak_Nm, '%7.0f'):>7s} {fmt(r.power_continuous_W and r.power_continuous_W / 1e3, '%9.0f'):>9s} "
          f"{fmt(w and w / rpm, '%8.0f'):>8s} {fmt(tau, '%7.1f'):>7s} "
          f"{fmt(r.power_continuous_W and r.power_continuous_W / r.mass_kg / 1e3, '%6.1f'):>6s} {'y' if r.fit else 'n':>4s} "
          f"{fmt(residual.get(r.name), '%8.2f'):>8s}")
print(f"\nfit: tau = {fit.torque_density_ref_Nm_kg:.2f} N m/kg x (w / {fit.speed_ref_rad_s:.0f} rad/s)^-{fit.exponent_speed:.3f}; "
      f"RMS log residual {fit.rms_log_residual:.3f} (x/ {np.exp(fit.rms_log_residual):.2f})")

In [ ]:
model_bare = DatabaseMassModel(specific_power_max_W_kg=20000.0, smoothing_kg=1e-6)
speed_rad_s = np.geomspace(100, 3000, 300)


def torque_density_model(w, inverter_W_kg=None):
    # Continuous torque density of a whole machine at base speed w (rating 1 N m continuous).
    machine = Motor(power_rated_W=w * 1.0, max_torque_Nm=2.0)
    model = replace(model_bare, specific_power_inverter_W_kg=inverter_W_kg)
    return 1.0 / model.mass_kg(machine)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
ax1.plot(speed_rad_s / rpm, [torque_density_model(w) for w in speed_rad_s], color=blue, lw=2,
         label="fit, bare (cap 20 kW/kg)")
ax1.plot(speed_rad_s / rpm, [torque_density_model(w, 20000.0) for w in speed_rad_s], color=blue, lw=1.2, ls="--",
         label="fit + inverter at 20 kW/kg")
ax1.plot(speed_rad_s / rpm, [1.0 / TorqueDensityMassModel(15.0, 10000.0, 1e-6).mass_kg(
    Motor(power_rated_W=w * 1.0, max_torque_Nm=2.0)) for w in speed_rad_s], color=ink_muted, lw=1, ls=":",
         label="Tier 13: 15 N m/kg peak (7.5 continuous), 10 kW/kg, integrated")
markers = dict(axial="o", radial="s", unknown="^")
for r in records:
    if r.torque_continuous_Nm is not None:
        w, tau = r.speed_base_rad_s(), r.torque_density_continuous_Nm_kg()
        color = blue if r.fit else orange
        ax1.scatter(w / rpm, tau, marker=markers[r.topology], color=color, edgecolor=ink, lw=0.5, zorder=3, s=36)
    else:
        w = r.speed_rated_rad_s or r.speed_max_rad_s
        ax1.scatter(w / rpm, 0.5 * r.torque_peak_Nm / r.mass_kg, marker=markers[r.topology], facecolor="none",
                    edgecolor=yellow, lw=1.5, zorder=3, s=40)
    ax1.annotate(r.name, (w / rpm, (r.torque_density_continuous_Nm_kg() or 0.5 * r.torque_peak_Nm / r.mass_kg)),
                 fontsize=6.5, color=ink_muted, xytext=(3, 2), textcoords="offset points")
ax1.scatter([], [], color=blue, label="fitted (bare / unknown)")
ax1.scatter([], [], color=orange, label="integrated (not fitted)")
ax1.scatter([], [], facecolor="none", edgecolor=yellow, label="peak only, 0.5 x peak (not fitted)")
ax1.set(xscale="log", yscale="log", xlabel="base speed P_cont / T_cont [rpm]",
        ylabel="continuous torque density [N m/kg]", title="Torque density against speed class")
ax1.legend(frameon=False, fontsize=7.5, loc="lower left")
names = [n for n, _ in fit.residuals]
ax2.barh(names, [x for _, x in fit.residuals], color=[orange if abs(np.log(x)) > np.log(1.5) else blue
                                                      for _, x in fit.residuals])
ax2.axvline(1.0, color=ink, lw=1)
ax2.set(xscale="log", xlabel="model mass / database mass", title="Fit residuals (fitted rows)")
plt.tight_layout()

check("defaults equal the re-fit (tau_ref, exponent)",
      abs(DatabaseMassModel().torque_density_ref_Nm_kg - fit.torque_density_ref_Nm_kg) < 0.01
      and abs(DatabaseMassModel().exponent_speed - fit.exponent_speed) < 0.001)
check("torque density falls with speed (0 < a < 1, so specific power still rises)", 0 < fit.exponent_speed < 1)
check("every fitted machine within a factor of 2.6", all(abs(np.log(x)) < np.log(2.6) for _, x in fit.residuals))
check("RMS log residual below 0.4", fit.rms_log_residual < 0.4)
check("Helix SPX242 family and EMRAX within 30 %",
      all(abs(x - 1) < 0.3 for n, x in fit.residuals if n.startswith(("SPX242", "EMRAX"))))
check("every row cites an https source", all(r.source_url.startswith("https://") for r in records))

**Reading the fit:**

- The scatter is large. At about 2,300-2,500 rpm, continuous torque density runs from 7.4 N m/kg (magni350,
  165 kW continuous of 350 kW peak) to 30 N m/kg (Siemens SP200D).
- The best low-speed machines are about 1.8 times the fleet fit: the SP200D at 30 N m/kg and the SP260D at
  20 N m/kg. So is the Evolito D250, at 25 N m/kg near 9,500 rpm.
- The exponent a = 0.27 is weak. At fixed power the mass falls as w^-0.73, so the data alone still favour fast
  machines until the specific-power cap binds.
- The cap is the Tier 15 bare-machine assumption (20 kW/kg). No fitted machine except the D250 reaches it, so the
  data do not constrain it. Section 6 shows its sensitivity.

## 2. Stacking

Evolito stacks D250 and D500 units into one propulsion unit (newatlas.com, evtol.news). Its "1x3 / 2x3" D1500
suffix is the winding-set arrangement, not a stack count (magneticsmag.com).

A stacked machine carries n times the torque of one stack at the same speed. With the fitted mass linear in torque
at fixed speed, n stacks weigh n times one stack, plus `mass_overhead_stack_kg` per stack (default 0, because the
fit is to whole machines). The stack count is relaxed (continuous): T_cont / T_stack_max. At the fitted level it
does not change the mass; it is reported.

The Helix SPX242 family is one lamination built at stack lengths of 50, 94 and 175 mm. A straight line through it
gives the cost of length scaling: a fixed mass (end windings, housing, bearings) plus an incremental torque
density.

In [ ]:
helix = [r for r in records if r.name in ("SPX242-50", "SPX242-94", "SPX242-175")]
torque = np.array([r.torque_continuous_Nm for r in helix]); mass = np.array([r.mass_kg for r in helix])
slope_kg_Nm, mass_fixed_kg = np.polyfit(torque, mass, 1)
print(f"Helix SPX242 family: mass = {mass_fixed_kg:.1f} kg + T_cont / {1 / slope_kg_Nm:.1f} N m/kg")

unit = rubber_machine(Motor, 200.0, 600.0, **ratios)
model = DatabaseMassModel(torque_continuous_max_stack_Nm=0.5 * 2.5 * 600.0, mass_overhead_stack_kg=1.5, smoothing_kg=1e-6)
plain = DatabaseMassModel(smoothing_kg=1e-6)
counts = np.linspace(1, 6, 26)
stacked_mass = [float(model.mass_kg(rubber_machine(Motor, 200.0, n * 600.0, **ratios))) for n in counts]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(counts, stacked_mass, color=blue, label="1.5 kg overhead per stack")
ax1.plot(counts, [float(plain.mass_kg(rubber_machine(Motor, 200.0, n * 600.0, **ratios))) for n in counts], color=aqua,
         label="no overhead (default)")
ax1.set(xlabel="stack count (relaxed)", ylabel="machine mass [kg]", title="Stacking at 200 rad/s base speed")
ax1.legend(frameon=False)
ax2.scatter(torque, mass, color=blue, zorder=3)
t_line = np.linspace(0, 600, 10)
ax2.plot(t_line, mass_fixed_kg + slope_kg_Nm * t_line, color=ink_muted, ls="--")
ax2.set(xlabel="continuous torque [N m]", ylabel="mass [kg]", title="Helix SPX242 stack lengths", xlim=(0, 620), ylim=(0, 60))
plt.tight_layout()

three = rubber_machine(Motor, 200.0, 1800.0, **ratios)
check("relaxed stack count = T_cont / T_stack_max", abs(float(model.count_stacks(three)) - 3.0) < 1e-12)
check("three stacks weigh three times one (no overhead)",
      abs(float(plain.mass_kg(three)) - 3 * float(plain.mass_kg(unit))) < 1e-4)
check("overhead adds per stack", abs(float(model.mass_kg(three) - plain.mass_kg(three)) - 4.5) < 1e-6)
check("length scaling has a positive fixed mass (Helix)", mass_fixed_kg > 0)

## 3. Gearbox stage count

Let x = ln(ratio) / ln(5).

- The staircase is n = 1 + sum_k sigmoid((x - k) / 0.02 - 3). Each step sits 3 widths past the integer, so one
  stage covers ratios up to 5 (n = 1.047 at 5:1).
- The relaxed form is a softplus max(1, x).
- Efficiency is 0.99 x 0.99^n: 0.980, 0.970 and 0.961.
- The mass factor is 1 + 0.3 (n - 1).

In Halo, the AFDD00 drive mass is taken at the XV-15 ratio (35.4:1, three stages here, where the transmission
factor is calibrated) and scaled by mass_factor(ratio) / mass_factor(35.4).

In [ ]:
stage = GearStageModel()
relaxed = GearStageModel(staircase=False)
ratio = np.geomspace(1.0, 200.0, 800)
n_stair = np.array([float(stage.count_stages(r)) for r in ratio])
n_relaxed = np.array([float(relaxed.count_stages(r)) for r in ratio])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ratio, np.maximum(1, np.ceil(np.log(ratio) / np.log(5) - 1e-12)), color=ink_muted, lw=1, ls=":", label="ceil")
ax1.plot(ratio, n_stair, color=blue, lw=2, label="smooth staircase")
ax1.plot(ratio, n_relaxed, color=orange, lw=1.5, label="relaxed (softplus)")
ax1.set(xscale="log", xlabel="overall ratio (fast / slow)", ylabel="stages", title="Stage count")
ax1.legend(frameon=False)
afdd_ratio_factor = (ratio / ratio_reference_gear_stages()) ** 0.09899
ax2.plot(ratio, [float(stage_mass_ratio(stage, r)) for r in ratio], color=blue, lw=2,
         label="staged mass / AFDD at 35.4:1")
ax2.plot(ratio, afdd_ratio_factor, color=ink_muted, lw=1.2, ls="--", label="AFDD00 alone (input-speed exponent 0.099)")
ax2b = ax2.twinx()
ax2b.plot(ratio, [float(stage.efficiency(r)) for r in ratio], color=aqua, lw=1.5, label="efficiency")
ax2b.set_ylabel("efficiency", color=aqua); ax2b.grid(False)
ax2.set(xscale="log", xlabel="overall ratio", ylabel="drive mass factor", title="Mass with stage steps, efficiency")
ax2.legend(frameon=False, loc="upper left")
plt.tight_layout()

check("one stage for ratio <= 5 (within 0.05)", all(abs(float(stage.count_stages(r)) - 1) < 0.05 for r in (1.0, 3.0, 5.0)))
check("staircase monotonic", np.all(np.diff(n_stair) >= -1e-12))
check("relaxed monotonic", np.all(np.diff(n_relaxed) >= -1e-12))
check("ceil away from the steps (6.5 -> 2, 24 -> 2, 36 -> 3, 100 -> 3)",
      all(abs(float(stage.count_stages(r)) - n) < 0.02 for r, n in ((6.5, 2), (24, 2), (36, 3), (100, 3))))
check("efficiency 0.970 at two stages (the Tier 13 constant)", abs(float(stage.efficiency(20.0)) - 0.970) < 1e-3)
check("stage model neutral at the XV-15 calibration ratio",
      abs(float(stage_mass_ratio(stage, ratio_reference_gear_stages())) - 1) < 1e-12)

opti = asb.Opti()
r_var = opti.variable(init_guess=8.0, lower_bound=1.5, upper_bound=40.0)
m_var = opti.variable(init_guess=1000.0, lower_bound=50.0, upper_bound=3000.0)
motor = rubber_machine(Motor, m_var, 4e5 / m_var, mass_model=DatabaseMassModel(specific_power_inverter_W_kg=2e4), **ratios)
opti.subject_to(m_var == 40.0 * r_var)
opti.minimize(motor.get_mass() + 100 * stage.mass_factor(r_var))
sol = opti.solve(verbose=False)
check("both models solve together inside asb.Opti", sol.value(motor.get_mass()) > 0)

## 4. The drive trade at the solved Halo point (no re-solve)

Hold the motor's rated power and the rotor speed of the fast-set solution with both options on (staircase stage
count), and move only the reduction ratio, which sets the motor speed. Two ways to count the drive mass (motor plus gearbox, per side):

- **plan 033:** the database machine with its inverter, plus the AFDD drive with explicit stages;
- **Tier 13:** 15 N m/kg peak and 10 kW/kg, plus AFDD00 with its own ratio exponent.

In [ ]:
both = replace(fast, machine_mass_model="database", gearbox_stages=True)             # relaxed stages (default)
both_staircase = replace(both, gear_stage_model=GearStageModel(staircase=True))
t0 = time.time()
sized_staircase = solve_halo_sizing(assumptions=both_staircase)
print(f"fast set, both on (staircase): {time.time() - t0:.0f} s, {sized_staircase.mass_takeoff_kg / u.lbm:,.0f} lb")
d = sized_staircase.design
power_motor_W = sized_staircase.power_rated_motor_W
power_gearbox_W = power_motor_W                         # thermal off: the gearbox carries the motor rating
speed_rotor_rad_s = d.speed_tip_m_s / sized_staircase.radius_rotor_m
factors = calibration_factors()
ratio_sweep = np.geomspace(2.0, 40.0, 300)


def drive_mass_kg(r, database=True):
    w = r * speed_rotor_rad_s
    machine = Motor(power_rated_W=power_motor_W, max_torque_Nm=2 * power_motor_W / w)
    if database:
        m_motor = DatabaseMassModel(specific_power_inverter_W_kg=20000.0).mass_kg(machine)
        m_gear = factors.transmission * afdd.mass_gearbox_rotor_shaft_afdd00_kg(
            2, 2 * power_gearbox_W, speed_rotor_rad_s * ratio_reference_gear_stages(), speed_rotor_rad_s
        ) * stage_mass_ratio(GearStageModel(), r) / 2
    else:
        m_motor = TorqueDensityMassModel(15.0, 10000.0).mass_kg(Motor(power_rated_W=power_motor_W,
                                                                       max_torque_Nm=2.5 * power_motor_W / (1.25 * w)))
        m_gear = factors.transmission * afdd.mass_gearbox_rotor_shaft_afdd00_kg(
            2, 2 * power_gearbox_W, w, speed_rotor_rad_s) / 2
    return float(m_motor), float(m_gear)


new = np.array([drive_mass_kg(r) for r in ratio_sweep])
old = np.array([drive_mass_kg(r, False) for r in ratio_sweep])
speed_motor_rpm = ratio_sweep * speed_rotor_rad_s / rpm
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(speed_motor_rpm, new.sum(axis=1), color=blue, lw=2.2, label="plan 033: motor + gearbox")
ax.plot(speed_motor_rpm, new[:, 0], color=blue, lw=1, ls="--", label="plan 033: motor (with inverter)")
ax.plot(speed_motor_rpm, new[:, 1], color=blue, lw=1, ls=":", label="plan 033: gearbox (staged)")
ax.plot(speed_motor_rpm, old.sum(axis=1), color=orange, lw=2.2, label="Tier 13: motor + gearbox")
ax.plot(speed_motor_rpm, old[:, 0], color=orange, lw=1, ls="--", label="Tier 13: motor")
best = np.argmin(new.sum(axis=1))
ax.scatter(speed_motor_rpm[best], new.sum(axis=1)[best], color=blue, zorder=4)
ax.axvline(d.speed_peak_motor_rad_s / rpm, color=ink_muted, lw=1)
ax.annotate("solved motor speed", (d.speed_peak_motor_rad_s / rpm, ax.get_ylim()[1] * 0.95), fontsize=8,
            color=ink_muted, xytext=(4, 0), textcoords="offset points")
for n in (1, 2, 3):
    ax.axvline(5 ** n * speed_rotor_rad_s / rpm, color=grid_color, lw=3, zorder=0)
ax.set(xscale="log", xlabel="motor base speed [rpm] (ratio x rotor speed)", ylabel="mass per side [kg]",
       title=f"Drive mass per side at {power_motor_W / 1e3:,.0f} kW rated, rotor {speed_rotor_rad_s / rpm:.0f} rpm")
ax.legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout()
print(f"lightest drive: ratio {ratio_sweep[best]:.1f}, motor {speed_motor_rpm[best]:,.0f} rpm, "
      f"{new.sum(axis=1)[best]:.0f} kg per side; at 8:1 {np.interp(8, ratio_sweep, new.sum(axis=1)):.0f} kg")
check("the stage steps appear in the gearbox mass", new[-1, 1] > new[0, 1])
check("slow (8:1) drive heavier than the solved one",
      np.interp(8, ratio_sweep, new.sum(axis=1)) > np.interp(d.reduction_ratio, ratio_sweep, new.sum(axis=1)))

## 5. Halo sizing, fast set: each option alone, both, and the ratio cap

Each case is a complete sizing (900 kg, 210 kt). The Halo default stage count is the relaxed (softplus) form, a
smooth max(1, x).

The staircase is shown alongside. With the staircase, `solve_halo_sizing` starts from the generic guess and from
the relaxed-stage solution, and keeps the lighter result. These are two starting points, not a loop: every stage
count has its own local optimum, for example a one-stage generator step-up near 6,300 rpm. The staircase solves
on this fast set, but with the database machines it fails through the AeroBuildup and thermal starts. That is why
Halo uses the relaxed form by default.

The ratio-cap sweep (relaxed stages) forces slower motors, and shows what a slow motor costs under the database
data. Each point also starts from the uncapped solution, and the lighter result is kept.

In [ ]:
cases = dict(reference=fast, database=replace(fast, machine_mass_model="database"),
             stages=replace(fast, gearbox_stages=True), both=both, **{"both (staircase)": both_staircase})
results = {"both (staircase)": sized_staircase}
for name, a in cases.items():
    if name not in results:
        results[name] = solve_halo_sizing(assumptions=a)


def summary(name, r, a):
    d, g, m = r.design, a.gear_stage_model, dict(r.powertrain_masses_kg)
    step_up = d.speed_peak_generator_rad_s / a.speed_output_turboshaft_rad_s
    return (f"{name:12s} {r.mass_takeoff_kg / u.lbm:7,.0f} lb | motor {d.speed_peak_motor_rad_s / rpm:6,.0f} rpm, "
            f"ratio {d.reduction_ratio:5.1f} ({float(g.count_stages(d.reduction_ratio)):.2f} st) | gen "
            f"{d.speed_peak_generator_rad_s / rpm:6,.0f} rpm, step-up {step_up:4.1f} ({float(g.count_stages(step_up)):.2f} st) | "
            f"motors {m['motor']:4.0f} kg, gearboxes {m['gearbox']:4.0f}, generators {m['generator']:4.0f}, "
            f"step-ups {m['generator_gearbox']:4.0f}")


for name, a in cases.items():
    print(summary(name, results[name], a))
for r in results.values():
    assert r.min_margin > -1e-6
check("every fast-set case closes with all margins", all(r.min_margin > -1e-6 for r in results.values()))
check("fast-set reference unchanged by the new code (13,702 lb)",
      abs(results["reference"].mass_takeoff_kg / u.lbm - 13702) < 10)

In [ ]:
caps = (4.9, 8.0, 12.0, 18.0, 40.0)
sweeps = {}
for label, base in (("database + stages", both),
                    ("database (10 kW/kg cap) + stages", replace(both, specific_power_max_machine_bare_W_kg=10000.0))):
    rows = []
    for cap in caps:
        # Starting points: the default ones, plus the uncapped both-on solution (its ratio clipped to the cap).
        candidates = []
        for start in (None, results["both"]):
            try:
                candidates.append(solve_halo_sizing(assumptions=replace(base, reduction_ratio_max=cap), initial=start,
                                                    max_iter=1500))
            except RuntimeError:
                pass
        if candidates:
            r = min(candidates, key=lambda c: c.mass_takeoff_kg)
            rows.append((cap, r))
            print(f"{label:34s} cap {cap:4.1f}: " + summary("", r, base)[13:])
        else:
            print(f"{label:34s} cap {cap:4.1f}: failed")
    sweeps[label] = rows

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
for (label, rows), color in zip(sweeps.items(), (blue, violet)):
    ax1.plot([r.design.reduction_ratio for _, r in rows], [r.mass_takeoff_kg / u.lbm for _, r in rows], "o-",
             color=color, label=label)
    ax2.plot([r.design.speed_peak_motor_rad_s / rpm for _, r in rows],
             [dict(r.powertrain_masses_kg)["motor"] + dict(r.powertrain_masses_kg)["gearbox"] for _, r in rows], "o-",
             color=color, label=label)
ax1.set(xlabel="solved rotor reduction ratio (capped)", ylabel="take-off mass [lb]",
        title="Cost of a slow motor (fast set)")
ax2.set(xlabel="solved motor speed [rpm]", ylabel="motors + rotor gearboxes, both sides [kg]",
        title="Drive mass against motor speed (sized aircraft)")
ax1.legend(frameon=False, fontsize=8); ax2.legend(frameon=False, fontsize=8)
plt.tight_layout()
db = sweeps["database + stages"]
check("slowest cap is heaviest with the database",
      max(db, key=lambda c: c[1].mass_takeoff_kg)[0] == min(c for c, _ in db))
check("staircase lighter than relaxed on the fast set (it credits the two-stage 25:1 limit)",
      results["both (staircase)"].mass_takeoff_kg < results["both"].mass_takeoff_kg)

## 6. The reference with both options on

Thermal on, AeroBuildup, AFDD wing, equivalent-circuit battery, 900 kg at 210 kt, with the relaxed stage count.
The solve takes several minutes; it chains the thermal-off, Scholz and constant-battery starts. The stage count
printed is the relaxed value, and a real gearbox has its ceiling.

In [ ]:
reference_both = HaloAssumptions(machine_mass_model="database", gearbox_stages=True)
t0 = time.time()
sized_ref = solve_halo_sizing(assumptions=reference_both)
print(f"solve {time.time() - t0:.0f} s")
print(summary("reference", sized_ref, reference_both))
d = sized_ref.design
print(f"take-off mass {sized_ref.mass_takeoff_kg / u.lbm:,.0f} lb against the 14,037 lb reference "
      f"({sized_ref.mass_takeoff_kg / u.lbm - 14037:+,.0f} lb)")
check("reference with both options closes with all margins", sized_ref.min_margin > -1e-6
      and abs(sized_ref.closure_residual_kg) < 1e-3)
check("reference with both options carries 900 kg", abs(sized_ref.mass_payload_kg - 900.0) < 1e-6)
masses = dict(sized_ref.powertrain_masses_kg)
x_rotor = np.log(d.reduction_ratio) / np.log(5.0)
print(f"motor {d.speed_peak_motor_rad_s / rpm:,.0f} rpm, rotor ratio {d.reduction_ratio:.1f}:1 "
      f"({float(reference_both.gear_stage_model.count_stages(d.reduction_ratio)):.2f} relaxed stages, "
      f"{int(np.ceil(x_rotor))} as built), motors {masses['motor']:.0f} kg and rotor gearboxes {masses['gearbox']:.0f} kg "
      f"(both sides), generators {masses['generator']:.0f} kg, step-ups {masses['generator_gearbox']:.0f} kg")

## 7. Summary

In [ ]:
failed = [name for name, passed in check_log if not passed]
print(f"{len(check_log) - len(failed)} / {len(check_log)} notebook checks passed")
for name in failed:
    print("  FAIL:", name)

In [ ]:
import os, unittest

os.chdir(repo_root)
suite = unittest.defaultTestLoader.discover(str(repo_root / "tests"), top_level_dir=str(repo_root / "tests"))
test_result = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(suite)
assert test_result.wasSuccessful() and not failed, "Verification failed"